In [9]:
from Bio import SeqIO
from subprocess import Popen, PIPE
from tqdm import tqdm
from joblib import Parallel, delayed

In [10]:
def read_fa(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id)
        seq = str(x.seq).replace("U", "T")
        res[id] = seq
    return res

def read_fa2(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id).split('|')[0]
        seq = str(x.seq) #.replace("U", "T")
        res[id] = seq
    return res

In [11]:
# Rutas de entrada
mir_dict_seq = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/miRNA.fa")
mir_dict_ss = read_fa2("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_2d/miRNA_2d_ss.fa")
mir_list = list(mir_dict_ss.keys())

In [12]:
def join_ss_seq(mir):
  
  seq = mir_dict_seq[mir]
  ss = mir_dict_ss[mir]

  path = f"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/seq+ss_join_mirna/seq+ss_join_{mir}.txt"

  f = open(path, "w")

  f.write(seq + "\n")
  f.write(ss)

  f.close()

In [13]:
for mir in mir_list:
  try:
    join_ss_seq(mir)
  except:
    print(mir)

In [15]:
def eval_rna_structure(mir):
  input = f"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/seq+ss_join_mirna/seq+ss_join_{mir}.txt"
  output = f"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/ss_loops_mirna/loops_{mir}.txt"
  
  command = f"RNAeval -v -d2 < {input} > {output}"
  result = Popen(command, shell=True, stdout=PIPE)
  eval_loops, err = result.communicate()

In [16]:
def worker(mir):
    try:
        eval_rna_structure(mir)
    except Exception as e:
        print(mir)

if __name__ == '__main__':
    num_cores = 8
    results = Parallel(n_jobs=num_cores)(delayed(worker)(lnc) for lnc in tqdm(mir_list))

  0%|          | 0/267 [00:00<?, ?it/s]/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
  6%|▌         | 16/267 [00:00<00:02, 86.22it/s]/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: /Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA: No such file or directory
/bin/sh: 

In [18]:
import csv
import os

mir_characterization_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/ss_loops_mirna/"
generated_file_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/miRNA_Precursor_features.csv"

files = os.listdir(mir_characterization_path)

sequences = {}
for file_name in files:
    name = file_name[6:21]
    file_path = os.path.join(mir_characterization_path, file_name)
    with open(file_path, 'r') as file:
        features = {"External": 0, "Interior": 0, "Hairpin": 0, "Multi": 0, "Energy": 0}
        lines = file.readlines()[:-2]
        for line in lines:
            tokens = line.split()
            features[tokens[0]] += 1
            features["Energy"] += int(tokens[-1])
        sequences[name] = features

sorted_sequences = dict(sorted(sequences.items()))
with open(generated_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    for seq in sorted_sequences.keys():
        features = sorted_sequences[seq]
        writer.writerow([seq, features["Interior"], features["Hairpin"], features["Multi"], float(features["Energy"]/100)])